# PerioVision AI: Priority 1 evaluation runs on Colab (no training)

This notebook **trains nothing**. It runs the already-trained, signed models on two test sets that are not on
the laptop and writes one row per film, so confidence intervals can be computed:

| Step | Data (downloaded inside Colab) | Output |
|---|---|---|
| DENTEX split audit | DENTEX `quadrant_enumeration`, the exact seed-42 split used for training | `split_audit_dentex.json` |
| DENTEX detector predictions | the 63 DENTEX test films | `dentex_detector_per_film.csv` |
| MM-OPG split audit | ToothXpert MM-OPG official train / test | `split_audit_mmopg.json` |
| MM-OPG screen predictions | the 450-film official test split | `mmopg_screen_per_film.csv` |

**Before you start (once, on the laptop):** upload the backend code and weights to Google Drive. In PowerShell,
from the `Dental_progression` folder:

```powershell
Compress-Archive -Path backend\app, backend\research, backend\scripts, backend\config, backend\weights, backend\keys\model_signing.pub, backend\requirements.txt -DestinationPath $HOME\Downloads\periovision_backend.zip -Force
```

Upload `periovision_backend.zip` (about 330 MB) to `MyDrive/PerioVision/p1/`. The private signing key is
**not** included and is not needed.

**Run:** `Runtime` → `Change runtime type` → **T4 GPU** → Save, then `Runtime` → **Run all**.
Takes about 20–30 minutes, mostly the DENTEX download (~11 GB).

**After:** download `MyDrive/PerioVision/p1_results/`, copy the two CSVs into
`docs/evidence/predictions/` and the two JSONs into `docs/evidence/`, then on the laptop run
`cd backend; python -m research.compute_ci`.

In [ ]:
# 1. GPU + Google Drive
import os, shutil, subprocess, glob, json, zipfile
if not shutil.which("nvidia-smi"):
    raise RuntimeError("NO GPU attached. Runtime > Change runtime type > T4 GPU > Save, then Run all again.")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/PerioVision"
OUT = f"{ROOT}/p1_results"
os.makedirs(OUT, exist_ok=True)
CODE_ZIP = f"{ROOT}/p1/periovision_backend.zip"
assert os.path.exists(CODE_ZIP), f"Upload periovision_backend.zip to {CODE_ZIP} first (see the top cell)."

In [ ]:
# 2. Unpack the backend and install its dependencies (Colab's own torch is kept)
B = "/content/backend"
if not os.path.exists(f"{B}/research/split_audit.py"):
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall("/content/_code")
    # Compress-Archive puts each listed folder at the top level of the zip
    os.makedirs(f"{B}/keys", exist_ok=True)
    for name in os.listdir("/content/_code"):
        src = f"/content/_code/{name}"
        dst = f"{B}/keys/{name}" if name.endswith(".pub") else f"{B}/{name}"
        shutil.move(src, dst)
reqs = [l for l in open(f"{B}/requirements.txt") if l.strip() and not l.startswith("#")
        and not l.lower().startswith(("torch", "numpy"))]
open("/content/reqs.txt", "w").writelines(reqs)
!pip -q install -r /content/reqs.txt pyyaml
print(sorted(os.listdir(B)))

In [ ]:
# 3. DENTEX: download, convert to YOLO and rebuild the EXACT training split (seed 42, same code as
#    train_periovision_colab.ipynb cell 4). If your Drive still has splits/dentex_split.json, it must match.
import random
from huggingface_hub import hf_hub_download
RAW, SEED = "/content/dentex", 42
os.makedirs(RAW, exist_ok=True)
if not glob.glob(f"{RAW}/**/*quadrant_enumeration*.json", recursive=True):
    z = hf_hub_download("ibrahimhamamci/DENTEX", "DENTEX/training_data.zip", repo_type="dataset", local_dir=RAW)
    with zipfile.ZipFile(z) as f:
        f.extractall(RAW, [m for m in f.namelist() if "quadrant_enumeration/" in m and "disease" not in m])
    os.remove(z)
json_path = [p for p in glob.glob(f"{RAW}/**/*.json", recursive=True)
             if "quadrant_enumeration" in os.path.basename(p) and "disease" not in p][0]
img_dir = os.path.join(os.path.dirname(json_path), "xrays")

coco = json.load(open(json_path))
q_name = {c["id"]: str(c["name"]) for c in coco["categories_1"]}
t_name = {c["id"]: str(c["name"]) for c in coco["categories_2"]}
FDI = [f"{q}{n}" for q in (1, 2, 3, 4) for n in range(1, 9)]
cls_of = {k: i for i, k in enumerate(FDI)}
imgs = {im["id"]: im for im in coco["images"]}
per_img = {}
for a in coco["annotations"]:
    fdi = q_name[a["category_id_1"]] + t_name[a["category_id_2"]]
    if fdi not in cls_of:
        continue
    im = imgs[a["image_id"]]; W, H = im["width"], im["height"]
    x, y, w, h = a["bbox"]
    per_img.setdefault(im["id"], []).append(f"{cls_of[fdi]} {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}")
names = sorted(imgs[i]["file_name"] for i in per_img)
random.Random(SEED).shuffle(names)
n = len(names); n_test = n_val = round(n * 0.10)
split = {"test": names[:n_test], "val": names[n_test:n_test+n_val], "train": names[n_test+n_val:]}
drive_split = f"{ROOT}/splits/dentex_split.json"
if os.path.exists(drive_split):
    saved = json.load(open(drive_split))
    assert {k: sorted(v) for k, v in saved.items()} == {k: sorted(v) for k, v in split.items()}, \
        "Rebuilt split differs from the one saved during training: STOP and check that the split file on Drive is the one used for training."
    print("split matches the one saved during training")
print({k: len(v) for k, v in split.items()})   # expected: test 63, val 63, train 508

DET = "/content/det"
by_name = {imgs[i]["file_name"]: i for i in per_img}
for part, files in split.items():
    os.makedirs(f"{DET}/images/{part}", exist_ok=True); os.makedirs(f"{DET}/labels/{part}", exist_ok=True)
    for fn in files:
        dst = f"{DET}/images/{part}/{fn}"
        if not os.path.exists(dst):
            shutil.copy(os.path.join(img_dir, fn), dst)
        open(f"{DET}/labels/{part}/{os.path.splitext(fn)[0]}.txt", "w").write("\n".join(per_img[by_name[fn]]) + "\n")
open(f"{DET}/dentex_fdi.yaml", "w").write(
    f"path: {DET}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
    + "".join(f"  {i}: '{k}'\n" for i, k in enumerate(FDI)))

In [ ]:
# 4. MM-OPG: official test labels + images (same files the screen model was trained / tested with)
import urllib.request
M = "/content/mmopg"; os.makedirs(M, exist_ok=True)
HF = "https://huggingface.co/datasets/jeffrey423/ToothXpert.MM-OPG-Annotations/resolve/main"
for name, dst in (("MM-OPG-Train-Raw.json", "mmopg_train.json"), ("MM-OPG-Test.json", "mmopg_test.json"),
                  ("images_resized_public.zip", "images.zip")):
    if not os.path.exists(f"{M}/{dst}"):
        print("downloading", name); urllib.request.urlretrieve(f"{HF}/{name}", f"{M}/{dst}")
if not os.path.exists(f"{M}/images"):
    zipfile.ZipFile(f"{M}/images.zip").extractall(f"{M}/images")
print(len(glob.glob(f"{M}/images/**/*.*", recursive=True)), "image files")

In [ ]:
# 5. Split audits (patient IDs are not published for either dataset: these check for duplicated films)
os.environ["DB_MODE"] = "demo"
def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=B, text=True, capture_output=True, env=os.environ)
    print(r.stdout[-4000:], r.stderr[-3000:])
    if r.returncode:
        raise RuntimeError("step failed - see the messages above")
run(f"python -m research.split_audit audit --dataset yolo --root {DET} --show-names --out {OUT}/split_audit_dentex.json")
run(f"python -m research.split_audit audit --dataset mmopg --root {M} --show-names --out {OUT}/split_audit_mmopg.json")

In [ ]:
# 6. Per-film predictions of the deployed, signature-checked models (inference only)
run(f"python -m research.export_predictions yolo-detector --images {DET}/images/test --labels {DET}/labels/test "
    f"--names {DET}/dentex_fdi.yaml --out {OUT}/dentex_detector_per_film.csv")
run(f"python -m research.export_predictions mmopg-screen --data {M} --out {OUT}/mmopg_screen_per_film.csv")
print("Done. Download", OUT, "and follow the 'After' step at the top.")
print(sorted(os.listdir(OUT)))